# Introduzione alla Modellazione Semantica con dbt

Questo notebook accompagna la lezione sulla **modellazione semantica** usando dbt e il database AdventureWorks.

## Obiettivi

1. Esplorare i dati grezzi
2. Comprendere il problema del **fanout**
3. Vedere come dbt risolve questi problemi
4. Confrontare query sbagliate vs corrette

In [ ]:
# Installazione dipendenze (se necessario)
# !uv sync

## 1. Esplorazione dei Dati Grezzi

Iniziamo esplorando i dati grezzi. I dati sono memorizzati in file CSV nella cartella `adventureworks/seeds/`.

In [ ]:
import duckdb
import pandas as pd

# Connessione al database
con = duckdb.connect('adventureworks/data/adventureworks.duckdb')

# Lista delle tabelle
print("Tabelle disponibili:")
tables = con.execute("SHOW TABLES").fetchall()
for t in tables:
    print(f"  - {t[0]}")

### 1.1 Clienti

In [ ]:
# Visualizza clienti
customers = con.execute("SELECT * FROM customers").df()
print("CLIENTI:")
print(customers.to_string(index=False))

### 1.2 Ordini

In [ ]:
# Visualizza ordini
orders = con.execute("SELECT * FROM orders").df()
print("ORDINI (status: 1=pending, 5=shipped, 6=cancelled):")
print(orders.to_string(index=False))

### 1.3 Righe Ordine (con sconti!)

In [ ]:
# Visualizza righe ordine
order_lines = con.execute("SELECT * FROM order_lines").df()
print("RIGHE ORDINE (nota: discount = percentuale sconto):")
print(order_lines.to_string(index=False))

## 2. Il Problema del Fanout

Il **fanout** è uno dei problemi più comuni nelle query SQL su dati transazionali. Succede quando fai un JOIN tra tabelle e poi un'aggregazione senza cure adeguate.

In [ ]:
# Esempio: Query sbagliata - conta le righe, non gli ordini
query_sbagliata = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(ol.order_id) AS order_count  -- ERRORE! Conta righe
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_sbagliato = con.execute(query_sbagliata).df()
print("❌ QUERY SBAGLIATA (fanout):")
print(result_sbagliato.to_string(index=False))

In [ ]:
# Query corretta - usa DISTINCT
query_corretta = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(DISTINCT o.order_id) AS order_count  -- CORRETTO!
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_corretto = con.execute(query_corretta).df()
print("\n⚠️ QUERY CORRETTA (con DISTINCT):")
print(result_corretto.to_string(index=False))

In [ ]:
# Confronto
print("\n📊 DIFFERENZA:")
comparison = result_sbagliato.merge(result_corretto, on='customer_id', suffixes=('_sbagliato', '_corretto'))
comparison['differenza'] = comparison['order_count_sbagliato'] - comparison['order_count_corretto']
print(comparison[['customer_name', 'order_count_sbagliato', 'order_count_corretto', 'differenza']].to_string(index=False))

## 3. Problema 2: Sconti e Revenue

Un altro problema comune: usare il campo `total` dell'ordine invece di calcolare il revenue dalle righe (dove ci sono gli sconti).

In [ ]:
# Query sbagliata: usa il campo total dell'header
query_header = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    SUM(o.total) AS lifetime_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 5  -- solo shipped
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_header = con.execute(query_header).df()
print("❌ QUERY SBAGLIATA (usa orders.total):")
print(result_header.to_string(index=False))

In [ ]:
# Query corretta: calcola da righe con sconti
query_lines = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    SUM(ol.line_total * (1 - ol.discount)) AS lifetime_value  -- considera sconti!
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
WHERE o.status = 5
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_lines = con.execute(query_lines).df()
print("\n⚠️ QUERY CORRETTA (calcola da order_lines con sconti):")
print(result_lines.to_string(index=False))

In [ ]:
# Confronto
print("\n📊 DIFFERENZA (gli sconti fanno la differenza!):")
comp = result_header.merge(result_lines, on='customer_id', suffixes=('_header', '_lines'))
comp['differenza'] = comp['lifetime_value_header'] - comp['lifetime_value_lines']
print(comp[['customer_name', 'lifetime_value_header', 'lifetime_value_lines', 'differenza']].to_string(index=False))

## 4. Esecuzione dei Modelli dbt

Ora vediamo come i modelli dbt risolvono questi problemi automaticamente.

In [ ]:
# Visualizza i modelli marts
print("📦 FACT TABLES CREATE DA DBT:\n")

# fct_customers
fct_customers = con.execute("SELECT * FROM fct_customers").df()
print("fct_customers:")
print(fct_customers.to_string(index=False))

In [ ]:
# fct_products
fct_products = con.execute("SELECT * FROM fct_products").df()
print("\nfct_products:")
print(fct_products.to_string(index=False))

In [ ]:
# fct_orders
fct_orders = con.execute("SELECT * FROM fct_orders").df()
print("\nfct_orders:")
print(fct_orders.to_string(index=False))

## 5. Confronto Finale

La modellazione semantica con dbt garantisce che:

1. ✅ I filtri corretti sono applicati (es. status = shipped)
2. ✅ Le aggregazioni usano DISTINCT dove necessario
3. ✅ I calcoli sono fatti dai dettagli, non dai campi aggregati
4. ✅ La logica è centralizzata e riutilizzabile

In [ ]:
# Esempio: query semplice come dovrebbe essere
print("✅ Con dbt, per ottenere i clienti per valore:")
print("\nSELECT first_name, last_name, lifetime_net_revenue")
print("FROM fct_customers")
print("ORDER BY lifetime_net_revenue DESC;\n")

simple = con.execute("SELECT first_name, last_name, lifetime_net_revenue FROM fct_customers ORDER BY lifetime_net_revenue DESC").df()
print("RISULTATO:")
print(simple.to_string(index=False))

## Esercizi Proposti

1. **Esercizio 1**: Quanti prodotti unici sono stati venduti per categoria?
2. **Esercizio 2**: Qual è il prodotto più scontato?
3. **Esercizio 3**: Quale città ha generato più revenue?

Provate a scrivere le query sia in SQL diretto che usando i modelli dbt!

In [ ]:
# Chiudi connessione
con.close()
print("✅ Notebook completato!")